In [1]:
import os
from pathlib import Path

import pandas as pd

Setting CWD to root

In [2]:
# __file__ = str(Path(".").absolute() / "interactions.ipynb")
# __file__
NB_DIR = Path(".").absolute() / "interactions.ipynb"
NB_DIR

PosixPath('/home/famo00001/kinodata-3D-affinity-prediction/prob/probing_notebooks/interactions.ipynb')

In [3]:
print(os.getcwd())
# _ROOT = Path(os.environ.get("HOME_PROJ_DIR", Path(__file__).resolve().parents[2]))
_ROOT = Path(os.environ.get("HOME_PROJ_DIR", NB_DIR.resolve().parents[2]))
os.chdir(_ROOT)
print(os.getcwd())

/home/famo00001/kinodata-3D-affinity-prediction/prob/probing_notebooks
/home/famo00001/kinodata-3D-affinity-prediction


In [9]:
interactions_dir = _ROOT / "data" / "plip_kinodata3d" / "processed"

mapping = pd.read_csv(_ROOT / "data" / "ident_to_activity_id.csv")
mapping.rename(columns={"activities.activity_id": "activity_id"}, inplace=True)

halogen_bonds_df = pd.read_csv(interactions_dir / "halogen_bonds.csv")
hydrogen_bonds_df = pd.read_csv(interactions_dir / "hydrogen_bonds.csv")
hydrophobic_interactions_df = pd.read_csv(interactions_dir / "hydrophobic_interactions.csv")
pi_stacking_df = pd.read_csv(interactions_dir / "pi_stacking.csv")
salt_bridges_df = pd.read_csv(interactions_dir / "salt_bridges.csv")


In [10]:
print(len(hydrogen_bonds_df))
hydrogen_bonds_df.head()

322499


,RESNR,RESTYPE,RESCHAIN,RESNR_LIG,RESTYPE_LIG,RESCHAIN_LIG,SIDECHAIN,DIST_H-A,DIST_D-A,DON_ANGLE,PROTISDON,DONORIDX,DONORTYPE,ACCEPTORIDX,ACCEPTORTYPE,LIGCOO,PROTCOO,activity_id
0,48,LEU,A,1,UNL,Z,False,1.79,2.69,151.89,False,1455,Npl,789,O2,"(-1.477, 18.513, 44.666)","(-3.405, 19.71, 46.118)",23227035
1,48,LEU,A,1,UNL,Z,False,1.92,2.90,162.57,True,784,Nam,1450,Nar,"(-0.942, 20.548, 43.637)","(-2.308, 22.328, 45.472)",23227035
2,52,SER,A,1,UNL,Z,False,2.03,2.96,151.87,True,845,Nam,1456,O2,"(-3.751, 12.62, 44.282)","(-4.813, 14.725, 42.494)",23227035
3,45,THR,A,1,UNL,Z,True,1.67,2.53,146.08,True,735,O3,1434,O2,"(1.909, 21.593, 40.698)","(3.132, 23.547, 41.736)",2111945
4,48,MET,A,1,UNL,Z,False,2.07,2.78,127.30,False,1432,Npl,782,O2,"(-1.082, 18.606, 45.341)","(-3.599, 19.474, 46.129)",2111945


In [11]:
num_hydrogen_bonds = hydrogen_bonds_df['activity_id'].value_counts().rename_axis('activity_id').reset_index(name='num_hydrogen_bonds')
num_hydrogen_bonds.head()

,activity_id,num_hydrogen_bonds
0,13302668,20
1,7808140,17
2,1097676,15
3,1090959,15
4,12688515,15


In [12]:
num_hydrogen_bonds = num_hydrogen_bonds.merge(mapping, on='activity_id', how='inner')
# num_hydrogen_bonds = num_hydrogen_bonds.set_index('index').sort_index()
print(len(num_hydrogen_bonds))
num_hydrogen_bonds.head()

112559


,activity_id,num_hydrogen_bonds,ident_processed,smiles_processed,y_processed,ident,compound_structures.canonical_smiles,activities.standard_value,similar.klifs_structure_id
0,13302668,20,33827,NC(=[NH2+])NCCC[C@H](NC(=O)[C@H](Cc1c[nH]c2ccc...,5.148742,33827,NC(=[NH2+])NCCC[C@H](NC(=O)[C@H](Cc1c[nH]c2ccc...,5.148742,983
1,7808140,17,25008,O=C(OC[C@H]1O[C@@H](OC(=O)c2cc(O)c(O)c(OC(=O)c...,6.425969,25008,O=C(OC[C@H]1O[C@@H](OC(=O)c2cc(O)c(O)c(OC(=O)c...,6.425969,14289
2,1097676,15,3102,Cc1nc(N)c2ncn([C@@H]3O[C@H](CO[P@@](=O)([O-])O...,8.585027,3102,Cc1nc(N)c2ncn([C@@H]3O[C@H](CO[P@@](=O)([O-])O...,8.585027,468
3,1090959,15,3069,Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO[P@@](=O)([O-])O...,8.136677,3069,Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO[P@@](=O)([O-])O...,8.136677,468
4,12688515,15,33006,CC(C)C[C@H](NC(=O)[C@H](Cc1c[nH]c2ccccc12)NC(=...,4.853872,33006,CC(C)C[C@H](NC(=O)[C@H](Cc1c[nH]c2ccccc12)NC(=...,4.853872,7181


In [14]:
print(num_hydrogen_bonds['ident_processed'].max())
print(num_hydrogen_bonds['ident'].max())

119710
119710


Check the SMILES and y of two Complexes by ident

In [8]:
from kinodata.data import KinodataDocked
from kinodata.transform import TransformToComplexGraph

dataset = KinodataDocked(transform=TransformToComplexGraph(remove_heterogeneous_representation=False), use_multiprocessing=True, num_processes= 16)
dataset

KinodataDocked(119522)

Choose idents to check:

Examine each:
    <br> Keeping in mind that we can access graphs by index and not ident. So we need to retrieve the index from the ident list

In [24]:
def check(dataset, bond_df, idents_list = [],
        num_samples = 5,
        ident_col_name = 'ident_processed',
        smiles_col_name = 'smiles_processed',
        activity_col_name = 'y_processed',
        ):
    """
    Checks if the info in merged dataframe by activity_id, matches the dataset:
        - checking ident
        - checking SMILES
    """
    # randome selection of n samples to check
    sample_idents = bond_df[ident_col_name].sample(num_samples, random_state=None).to_list()

    if not idents_list:
        print("Getting idents list")
        idents_list = dataset.data.ident.tolist() 
    for ident in sample_idents:
        # We can access the graphs in the dataset by index and not by ident, so:
        idx = idents_list.index(ident)
        print(f"Checking {ident= }, {idx= }")

        # Check SMILES
        assert dataset[idx].smiles == bond_df.loc[bond_df[ident_col_name] == ident, smiles_col_name].iloc[0], f"at {ident=} SMILES not matching"

        # Check activity value
        assert dataset[idx].y.item() == bond_df.loc[bond_df[ident_col_name] == ident, activity_col_name].iloc[0], f"at {ident=} activity value not matching"

In [17]:
idents_list = dataset.data.ident.tolist() 

/opt/conda/lib/python3.10/site-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [28]:
check(dataset, num_hydrogen_bonds, idents_list)

Checking ident= 86723, idx= 30427
Checking ident= 49159, idx= 5591
Checking ident= 46981, idx= 99824
Checking ident= 82832, idx= 76922
Checking ident= 32383, idx= 39855


In [27]:
print(len(mapping))

41238
